# Notebook 01 — Data Loading and Validation

**Project:** KPI-RAG: Explainable Root-Cause Analysis for 5G Networks  
**Stage:** Phase 1 — Data Loading  
**Purpose:** Load the TelecomTS dataset from local disk, validate its structure, understand the column schema, and save a clean copy in Parquet format for faster reloading in subsequent notebooks.

---

**Dataset:** TelecomTS (Feng et al., ICML 2026)  
**Source file:** `telecomts.csv` (1.22 GB)  
**Expected shape:** 32,000 rows × 9 columns  

**Columns:**
| Column | Type | Description |
|---|---|---|
| `start_time` | datetime | Window start timestamp |
| `end_time` | datetime | Window end timestamp |
| `sampling_rate` | int | Measurement resolution in ms (expected: 10) |
| `KPIs` | str (nested) | 16 numerical KPI channels, 128 timesteps each |
| `description` | str | Natural language summary of KPI behavior |
| `anomalies` | str (nested) | Anomaly flag, type, root cause, and troubleshooting ticket |
| `statistics` | str (nested) | Pre-computed per-channel statistics (mean, variance, trend, periodicity) |
| `labels` | str (nested) | Zone, application type, mobility, congestion, anomaly_present |
| `QnA` | str (nested) | Structured Q&A pairs with reasoning traces |

---

**Important:** The `statistics`, `labels`, `anomalies`, `description`, and `QnA` columns are excluded from classifier inputs to prevent label leakage. Only `KPIs` and derived features are used in Phases 2 and 3.

## Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import re
import warnings

warnings.filterwarnings('ignore')

print(f"pandas version:  {pd.__version__}")
print(f"numpy version:   {np.__version__}")

## Cell 2 — Load Dataset

The CSV uses comma separation and contains nested Python objects (arrays, dicts) stored as strings.  
`quoting=1` (QUOTE_ALL) is required to prevent pandas from splitting nested commas inside field values.  
`on_bad_lines='skip'` drops malformed rows silently — a known issue with a small number of rows in this file.

In [ ]:
DATA_CSV = r"C:\Users\DELL\Desktop\kpi_rag\data\telecomts.csv"

file_size_gb = os.path.getsize(DATA_CSV) / 1e9
print(f"File size: {file_size_gb:.2f} GB")
print("Loading dataset...")

df = pd.read_csv(
    DATA_CSV,
    sep=',',
    quoting=1,           # QUOTE_ALL — required for nested array fields
    on_bad_lines='skip', # drops ~8 malformed rows in this dataset
    engine='python',
    encoding='utf-8'
)

print(f"Rows:         {len(df):,}")
print(f"Columns:      {len(df.columns)}")
print(f"Column names: {df.columns.tolist()}")

## Cell 3 — Validate Shape

Expected: 32,000 rows and exactly 9 columns.  
A lower row count indicates rows were dropped due to malformed content — acceptable up to ~50 rows.

In [ ]:
expected_rows = 32000
expected_cols = 9

row_diff = expected_rows - len(df)

print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows:   {len(df):,}")
print(f"Dropped rows:  {row_diff}")

if row_diff > 50:
    print("WARNING: More than 50 rows dropped — inspect the CSV file.")
else:
    print("Row count acceptable.")

if len(df.columns) != expected_cols:
    print(f"WARNING: Expected {expected_cols} columns, got {len(df.columns)}.")
else:
    print("Column count correct.")

## Cell 4 — Time Range and Sampling Rate

The dataset timestamps extend to 2037 — this is expected.  
TelecomTS concatenates multiple recording sessions with sequentially assigned timestamps rather than real calendar dates.  
Timestamps are not used as features in the classifier.

In [ ]:
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time']   = pd.to_datetime(df['end_time'])

print(f"Start: {df['start_time'].min()}")
print(f"End:   {df['start_time'].max()}")
print(f"Note:  Extended range is expected — timestamps are synthetic session markers.")
print(f"Sampling rate: {df['sampling_rate'].mode()[0]} ms")

## Cell 5 — Anomaly Distribution

Expected class distribution: approximately 96% Normal, 4% Anomalous.  
This 96/4 imbalance is real — it reflects the natural rarity of faults in production networks.  
It drives two design decisions: (1) recording-level stratified split, (2) 80/20 rebalancing for Model 1 training.

In [ ]:
def get_anomaly_type(text):
    match = re.search(r"'type':\s*'([^']+)'", str(text))
    return match.group(1) if match else 'Normal'

def get_anomaly_status(text):
    return 'Anomaly' if "'exists': True" in str(text) else 'Normal'

df['anomaly_type']   = df['anomalies'].apply(get_anomaly_type)
df['anomaly_status'] = df['anomalies'].apply(get_anomaly_status)

print("Anomaly status:")
print(df['anomaly_status'].value_counts())
print(f"\nAnomaly rate: {(df['anomaly_status'] == 'Anomaly').mean() * 100:.1f}%")

print("\nAnomaly types:")
print(df['anomaly_type'].value_counts())
print(f"\nUnique types (including Normal): {df['anomaly_type'].nunique()}")

## Cell 6 — Inspect One Normal Sample

Displays the full content of one normal sample to confirm the column structure before feature extraction.

In [ ]:
normal_sample = df[df['anomaly_status'] == 'Normal'].iloc[0]

print("--- Normal Sample ---")
print(f"start_time:    {normal_sample['start_time']}")
print(f"sampling_rate: {normal_sample['sampling_rate']} ms")
print(f"\nKPIs (first 200 chars):")
print(str(normal_sample['KPIs'])[:200])
print(f"\ndescription (first 200 chars):")
print(str(normal_sample['description'])[:200])
print(f"\nanomalies:")
print(str(normal_sample['anomalies'])[:200])

## Cell 7 — Inspect One Anomalous Sample

Displays the full content of one anomalous sample including the troubleshooting ticket.  
The ticket fields (issue, symptoms, root_cause, resolution) will be used as the RAG retrieval corpus in Phase 5.  
They are excluded from classifier inputs.

In [ ]:
anomalous_sample = df[df['anomaly_status'] == 'Anomaly'].iloc[0]

print("--- Anomalous Sample ---")
print(f"start_time:    {anomalous_sample['start_time']}")
print(f"anomaly_type:  {anomalous_sample['anomaly_type']}")
print(f"\nKPIs (first 200 chars):")
print(str(anomalous_sample['KPIs'])[:200])
print(f"\nanomalies (first 500 chars):")
print(str(anomalous_sample['anomalies'])[:500])
print(f"\nQnA (first 300 chars):")
print(str(anomalous_sample['QnA'])[:300])

## Cell 8 — Extract KPI Channel Names

Confirms the 16 numerical KPI channels present in the dataset.  
These are the channels used to build the 582-dimensional feature vector in Phase 2.

In [ ]:
def get_kpi_channel_names(kpi_text):
    matches = re.findall(r"'(\w+)':\s*array", str(kpi_text))
    return matches

kpi_channels = get_kpi_channel_names(df['KPIs'].iloc[0])

print(f"KPI channels found: {len(kpi_channels)}")
for i, ch in enumerate(kpi_channels, 1):
    print(f"  {i:2}. {ch}")

## Cell 9 — Missing Values

Checks for missing values across all columns.  
The nested columns (KPIs, anomalies, statistics, labels, QnA) may show zero nulls because missing content is represented as empty strings, not NaN.

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

result = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
has_missing = result[result['Missing Count'] > 0]

if len(has_missing) == 0:
    print("No missing values found.")
else:
    print("Columns with missing values:")
    print(has_missing.to_string())

## Cell 10 — Confirm Troubleshooting Tickets Exist

Counts how many anomalous rows contain a troubleshooting ticket.  
Expected: all 1,235 anomalous rows have a ticket.  
These tickets form the RAG retrieval corpus in Phase 5.

In [ ]:
anomalous_df = df[df['anomaly_status'] == 'Anomaly']

has_ticket = anomalous_df['anomalies'].apply(
    lambda x: 'troubleshooting_tickets' in str(x)
).sum()

print(f"Anomalous samples:         {len(anomalous_df):,}")
print(f"Samples with tickets:      {has_ticket:,}")
print(f"Samples without tickets:   {len(anomalous_df) - has_ticket:,}")

if has_ticket == len(anomalous_df):
    print("All anomalous samples have troubleshooting tickets.")
else:
    print("WARNING: Some anomalous samples are missing tickets — inspect before Phase 5.")

## Cell 11 — Save to Parquet

Saves the loaded and annotated dataframe to Parquet format.  
Parquet loads approximately 10x faster than CSV and preserves column types.  
All subsequent notebooks load from this file instead of the raw CSV.

In [ ]:
OUTPUT_PATH = r"C:\Users\DELL\Desktop\kpi_rag\data\telecomts_full.parquet"

df.to_parquet(OUTPUT_PATH, index=False)

saved_size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Saved to: {OUTPUT_PATH}")
print(f"File size: {saved_size_mb:.1f} MB")
print(f"Rows saved: {len(df):,}")

## Cell 12 — Validation Summary

Final check confirming all validation criteria before proceeding to Phase 2.

In [ ]:
checks = {
    'Row count acceptable (>= 31,950)':    len(df) >= 31950,
    'Column count correct (9)':            len(df.columns) == 9,
    'Sampling rate is 10 ms':              df['sampling_rate'].mode()[0] == 10,
    'Multiple anomaly types (>= 11)':      df['anomaly_type'].nunique() >= 11,
    'Anomaly rate between 3% and 5%':      0.03 <= (df['anomaly_status'] == 'Anomaly').mean() <= 0.05,
    'All anomalous samples have tickets':  has_ticket == len(anomalous_df),
    'Parquet file saved':                  os.path.exists(OUTPUT_PATH),
}

all_passed = True
for check, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    print(f"[{status}] {check}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("All checks passed. Proceed to Notebook 02.")
else:
    print("One or more checks failed. Investigate before proceeding.")